# Ablation - CAD only (no PSG)
`use_encoder_prompt=False`: uses only the Conditioned Attention Decoder.

In [ ]:
import os, sys, shutil
import gdown

BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
os.chdir(BASE)

REPO = 'https://github.com/ThongLuc2k3/PGA_Unet2D.git'
REPO_ROOT = f'{BASE}/PGA_Unet2D'
PGA_ROOT = f'{REPO_ROOT}/Source/Prompt-Guided-XRay-Segmentation'
DATASET_NAME = 'dataset_FracAtlas'
DATASET_ID = '1sfMPFQvADmZLCPJC3xPyYDrblnFZ4kQv'
DS_ZIP = f'{BASE}/{DATASET_NAME}.zip'
DS_PATH = f'{PGA_ROOT}/{DATASET_NAME}'

if os.path.exists(REPO_ROOT):
    shutil.rmtree(REPO_ROOT)
!git clone -q --branch main --single-branch {REPO} {REPO_ROOT}
if PGA_ROOT not in sys.path:
    sys.path.insert(0, PGA_ROOT)

if not os.path.exists(DS_ZIP):
    gdown.download(f'https://drive.google.com/uc?id={DATASET_ID}', DS_ZIP, quiet=False)
if os.path.exists(DS_PATH):
    shutil.rmtree(DS_PATH)
!unzip -oq {DS_ZIP} -d {PGA_ROOT}/

os.makedirs(f'{PGA_ROOT}/checkpoints', exist_ok=True)
os.chdir(PGA_ROOT)
!pip install -q tqdm opencv-python timm scipy gdown
print(f'✅ Setup completed | base={BASE} | dataset={DATASET_NAME}')


## Ablation training via the shared `train.py`

**Configuration: CAD only (no PSG).** CAD decoder attention only, PSG encoder gate disabled (`USE_ENCODER_PROMPT=0`). QualityHead stays on so this row matches the main PGA-UNet training objective.

Every ablation configuration is trained through the same `train.py` as the full PGA-UNet, so batch size, optimizer, scheduler, gradient clipping, the 150-epoch / patience-15 early stopping, and **image-level merged `center_shift` validation for checkpoint selection** are identical across the study. Repeat the training cell with `SEED = 1`, `2`, `3` for the multi-seed ablation; `RUN_TAG` keeps the per-seed checkpoints from overwriting each other.

For test metrics and qualitative panels, run the matching notebook under `Source/File_Test/fracatlas/Ablation/`.

In [ ]:
# Repeat with SEED = 1, 2, 3 ... for the multi-seed ablation.
SEED = 22120196
RUN_TAG = f"seed{SEED}"

import os, glob
os.chdir(PGA_ROOT)
os.environ.update({
    "PROMPT_DATASET_ROOT": "dataset_FracAtlas",
    "PROMPT_IMG_SIZE": "512",
    "PROMPT_MODE": "center_mixed",
    "PROMPT_SCALE_FACTOR": "3.0",
    "PROMPT_SHIFT_RATIO": "0.5",
    "PROMPT_MIXED_SHIFT_PROB": "0.8",
    "PROMPT_EPOCHS": "150",
    "PROMPT_SEED": str(SEED),
    "RUN_TAG": RUN_TAG,
    "PROMPT_VARIANT": "full",
    "USE_ENCODER_PROMPT": "0",
    "BINARY_PROMPT": "0",
    "USE_QUALITY_HEAD": "1",
})

!python train.py

print(sorted(glob.glob(f"checkpoints/pga_unet_center_mixed_x3_shift05_qhead_nopsg_512_{RUN_TAG}_best.pth")))
